# 04 — Advanced Methods

Compare four advanced SBI methods against basic rejection ABC:

1. **Regression-adjusted ABC** (Beaumont et al. 2002)
2. **ABC-MCMC** (Marjoram et al. 2003)
3. **SMC-ABC** (Beaumont et al. 2009)
4. **Synthetic Likelihood** (Wood, 2010)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import corner
import time

from data_loader import load_all
from summary_stats import compute_observed_summaries, compute_summaries, SUMMARY_NAMES, IDX_ALL
from abc_rejection import run_rejection_abc, accept_quantile, PRIOR_BOUNDS
from abc_regression import regression_adjust
from abc_mcmc import run_abc_mcmc, effective_sample_size
from smc_abc import run_smc_abc
from synthetic_likelihood import run_synthetic_likelihood_mcmc
from simulator import simulate_fast

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# Load data
inf_ts, rew_ts, deg_hist = load_all()
s_obs, s_per_rep = compute_observed_summaries(inf_ts, rew_ts, deg_hist)

# Warm up Numba
_ = simulate_fast(0.2, 0.1, 0.3, seed=0)

## 4.0 Baseline: Rejection ABC

Load or run rejection ABC as the baseline for comparison.

In [ ]:
N_SIM = 50_000

results_file = '../results/rejection_abc_50k.npz'
if os.path.exists(results_file):
    print("Loading saved rejection ABC results...")
    data = np.load(results_file)
    thetas = data['thetas']
    distances = data['distances']
    summaries = data['summaries']
    mads = data['mads']
else:
    print("Running rejection ABC...")
    t0 = time.time()
    thetas, distances, summaries, mads = run_rejection_abc(
        s_obs=s_obs, n_sim=N_SIM, simulator_fn=simulate_fast,
        indices=IDX_ALL, seed=42, verbose=True,
    )
    elapsed = time.time() - t0
    print(f'Completed in {elapsed:.1f}s')
    os.makedirs('../results', exist_ok=True)
    np.savez(results_file, thetas=thetas, distances=distances, summaries=summaries, mads=mads, s_obs=s_obs)

# Baseline accepted samples at q=1%
acc_rej, acc_d_rej, acc_s_rej, thr_rej = accept_quantile(thetas, distances, summaries, quantile=0.01)
print(f"\nRejection ABC baseline: {len(acc_rej)} accepted, threshold={thr_rej:.3f}")
print(f"  Posterior medians: beta={np.median(acc_rej[:,0]):.4f}, gamma={np.median(acc_rej[:,1]):.4f}, rho={np.median(acc_rej[:,2]):.4f}")

## 4.1 Regression-Adjusted ABC (Beaumont et al. 2002)

Post-process the rejection ABC accepted samples using local weighted linear
regression to correct for the gap between simulated and observed summaries.
This sharpens posteriors without additional simulations.

In [ ]:
# Use a slightly larger acceptance quantile for regression adjustment (more data for fitting)
acc_reg, acc_d_reg, acc_s_reg, thr_reg = accept_quantile(thetas, distances, summaries, quantile=0.05)
print(f"Using {len(acc_reg)} samples (q=5%) for regression adjustment")

t0 = time.time()
adjusted_thetas = regression_adjust(acc_reg, acc_s_reg, acc_d_reg, s_obs)
elapsed = time.time() - t0
print(f"Regression adjustment took {elapsed:.3f}s")

# Clip to prior bounds (regression can push values outside)
adjusted_thetas[:, 0] = np.clip(adjusted_thetas[:, 0], *PRIOR_BOUNDS['beta'])
adjusted_thetas[:, 1] = np.clip(adjusted_thetas[:, 1], *PRIOR_BOUNDS['gamma'])
adjusted_thetas[:, 2] = np.clip(adjusted_thetas[:, 2], *PRIOR_BOUNDS['rho'])

print(f"Adjusted posterior medians: beta={np.median(adjusted_thetas[:,0]):.4f}, "
      f"gamma={np.median(adjusted_thetas[:,1]):.4f}, rho={np.median(adjusted_thetas[:,2]):.4f}")

# Save
np.savez('../results/regression_abc.npz', adjusted_thetas=adjusted_thetas)

## 4.2 ABC-MCMC (Marjoram et al. 2003)

Run a Markov chain within the ABC framework. The proposal is a Gaussian
random walk with covariance tuned from the rejection ABC accepted samples.

In [ ]:
# Tune proposal from rejection ABC accepted samples
proposal_cov = np.cov(acc_rej.T) * 0.5  # scale down for reasonable acceptance rate

# Use rejection ABC threshold as epsilon, and median accepted sample as init
theta_init = np.median(acc_rej, axis=0)
epsilon = thr_rej * 1.2  # slightly more generous than rejection threshold

print(f"ABC-MCMC settings:")
print(f"  epsilon = {epsilon:.4f}")
print(f"  theta_init = {theta_init}")
print(f"  proposal_cov diag = {np.diag(proposal_cov)}")

N_MCMC = 30_000

t0 = time.time()
chain, dist_chain, acc_rate = run_abc_mcmc(
    s_obs=s_obs,
    mads=mads,
    epsilon=epsilon,
    n_iter=N_MCMC,
    simulator_fn=simulate_fast,
    theta_init=theta_init,
    proposal_cov=proposal_cov,
    indices=IDX_ALL,
    seed=123,
    verbose=True,
)
elapsed = time.time() - t0
print(f"\nABC-MCMC: {N_MCMC} iterations in {elapsed:.1f}s, acceptance rate: {acc_rate:.4f}")

# Discard burn-in
burn_in = 5000
chain_post = chain[burn_in:]

# Thin by autocorrelation
ess = effective_sample_size(chain_post)
print(f"ESS: beta={ess[0]:.0f}, gamma={ess[1]:.0f}, rho={ess[2]:.0f}")

print(f"Posterior medians: beta={np.median(chain_post[:,0]):.4f}, "
      f"gamma={np.median(chain_post[:,1]):.4f}, rho={np.median(chain_post[:,2]):.4f}")

# Save
np.savez('../results/abc_mcmc.npz', chain=chain, chain_post=chain_post, acc_rate=acc_rate)

### ABC-MCMC Diagnostics: Trace Plots

In [ ]:
param_labels = [r'$\beta$', r'$\gamma$', r'$\rho$']

fig, axes = plt.subplots(3, 2, figsize=(14, 8))
for i in range(3):
    # Trace plot
    axes[i, 0].plot(chain[:, i], alpha=0.5, linewidth=0.5, color='steelblue')
    axes[i, 0].axvline(burn_in, color='red', linestyle='--', alpha=0.7, label='Burn-in')
    axes[i, 0].set_ylabel(param_labels[i])
    axes[i, 0].legend(fontsize=8)
    if i == 0:
        axes[i, 0].set_title('Trace plots')

    # Marginal posterior
    axes[i, 1].hist(chain_post[:, i], bins=40, density=True, color='steelblue', alpha=0.7, edgecolor='white')
    axes[i, 1].set_xlabel(param_labels[i])
    if i == 0:
        axes[i, 1].set_title('Posterior marginals (post burn-in)')

axes[2, 0].set_xlabel('Iteration')
plt.suptitle('ABC-MCMC Diagnostics', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/abc_mcmc_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.3 SMC-ABC (Beaumont et al. 2009)

Sequential Monte Carlo with adaptive tolerance schedule. Particles are
resampled and perturbed at each generation, progressively tightening the
acceptance threshold.

In [ ]:
t0 = time.time()
smc_particles, smc_weights, smc_epsilons, smc_all_particles = run_smc_abc(
    s_obs=s_obs,
    mads=mads,
    n_particles=1000,
    n_generations=10,
    simulator_fn=simulate_fast,
    indices=IDX_ALL,
    alpha=0.5,
    min_epsilon=0.5,
    seed=456,
    verbose=True,
)
elapsed = time.time() - t0
print(f"\nSMC-ABC: {len(smc_epsilons)} generations in {elapsed:.1f}s")
print(f"Final epsilon: {smc_epsilons[-1]:.4f}")
print(f"Weighted posterior medians: beta={np.average(smc_particles[:,0], weights=smc_weights):.4f}, "
      f"gamma={np.average(smc_particles[:,1], weights=smc_weights):.4f}, "
      f"rho={np.average(smc_particles[:,2], weights=smc_weights):.4f}")

# Save
np.savez('../results/smc_abc.npz',
         particles=smc_particles, weights=smc_weights, epsilons=smc_epsilons)

### SMC-ABC Tolerance Schedule

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(len(smc_epsilons)), smc_epsilons, 'o-', color='steelblue', markersize=8)
ax.set_xlabel('Generation')
ax.set_ylabel('Tolerance (epsilon)')
ax.set_title('SMC-ABC: Adaptive Tolerance Schedule')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/smc_tolerance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.4 Synthetic Likelihood (Wood, 2010)

Instead of comparing summaries through a distance function with a tolerance
(as in ABC), synthetic likelihood assumes summary statistics are multivariate
normal given the parameters:

$$s \mid \theta \sim \mathcal{N}(\mu_\theta, \Sigma_\theta)$$

At each MCMC step, we simulate $N_r$ replicates at the proposed $\theta$,
estimate $\hat\mu$ and $\hat\Sigma$ from these replicates, and evaluate
the MVN log-likelihood of $s_\text{obs}$. This constructs an approximate
likelihood that can be explored via standard Metropolis-Hastings MCMC.

Uses Campbell's (1980) robust covariance estimation with QR preconditioning,
following Wood's reference R implementation.

In [ ]:
# Tune proposal from rejection ABC (same as ABC-MCMC)
sl_proposal_cov = np.cov(acc_rej.T) * 0.3  # slightly smaller steps (each eval is expensive)
sl_theta_init = np.median(acc_rej, axis=0)

N_SL_ITER = 5_000
N_SIM_PER_EVAL = 200  # replicates per synthetic likelihood evaluation

print(f"Synthetic Likelihood MCMC settings:")
print(f"  n_iter = {N_SL_ITER}")
print(f"  n_sim_per_eval = {N_SIM_PER_EVAL}")
print(f"  theta_init = {sl_theta_init}")
print(f"  proposal_cov diag = {np.diag(sl_proposal_cov)}")

t0 = time.time()
sl_chain, sl_ll_chain, sl_acc_rate = run_synthetic_likelihood_mcmc(
    s_obs=s_obs,
    n_iter=N_SL_ITER,
    n_sim_per_eval=N_SIM_PER_EVAL,
    simulator_fn=simulate_fast,
    theta_init=sl_theta_init,
    proposal_cov=sl_proposal_cov,
    seed=789,
    verbose=True,
)
elapsed = time.time() - t0
print(f"\nSynthetic Likelihood: {N_SL_ITER} iterations in {elapsed:.1f}s")
print(f"  Total simulations: {N_SL_ITER * N_SIM_PER_EVAL:,}")
print(f"  Acceptance rate: {sl_acc_rate:.4f}")

# Burn-in
sl_burn_in = 1000
sl_chain_post = sl_chain[sl_burn_in:]

print(f"Posterior medians: beta={np.median(sl_chain_post[:,0]):.4f}, "
      f"gamma={np.median(sl_chain_post[:,1]):.4f}, rho={np.median(sl_chain_post[:,2]):.4f}")

# Save
np.savez('../results/synthetic_likelihood.npz',
         chain=sl_chain, ll_chain=sl_ll_chain, chain_post=sl_chain_post, acc_rate=sl_acc_rate)

### Synthetic Likelihood Diagnostics

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 8))
for i in range(3):
    # Trace plot
    axes[i, 0].plot(sl_chain[:, i], alpha=0.5, linewidth=0.5, color='purple')
    axes[i, 0].axvline(sl_burn_in, color='red', linestyle='--', alpha=0.7, label='Burn-in')
    axes[i, 0].set_ylabel(param_labels[i])
    axes[i, 0].legend(fontsize=8)
    if i == 0:
        axes[i, 0].set_title('Trace plots')

    # Marginal posterior
    axes[i, 1].hist(sl_chain_post[:, i], bins=40, density=True, color='purple', alpha=0.7, edgecolor='white')
    axes[i, 1].set_xlabel(param_labels[i])
    if i == 0:
        axes[i, 1].set_title('Posterior marginals (post burn-in)')

axes[2, 0].set_xlabel('Iteration')

# Log synthetic likelihood trace
fig2, ax2 = plt.subplots(figsize=(12, 3))
ax2.plot(sl_ll_chain, alpha=0.5, linewidth=0.5, color='purple')
ax2.axvline(sl_burn_in, color='red', linestyle='--', alpha=0.7)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Log synthetic likelihood')
ax2.set_title('Synthetic Likelihood Trace')

plt.tight_layout()
plt.savefig('../figures/synth_lik_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.5 Comparison: All Methods

Overlay posterior marginals from all five methods and compare quantitatively.

In [ ]:
methods = {
    'Rejection ABC (q=1%)': acc_rej,
    'Reg. Adjusted ABC': adjusted_thetas,
    'ABC-MCMC': chain_post,
    'SMC-ABC': smc_particles,
    'Synthetic Likelihood': sl_chain_post,
}
method_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for col in range(3):
    ax = axes[col]
    for i, (name, samples) in enumerate(methods.items()):
        if name == 'SMC-ABC':
            ax.hist(samples[:, col], bins=30, density=True, alpha=0.35,
                    color=method_colors[i], label=name, weights=smc_weights,
                    histtype='stepfilled')
        else:
            ax.hist(samples[:, col], bins=30, density=True, alpha=0.35,
                    color=method_colors[i], label=name, histtype='stepfilled')
    ax.set_xlabel(param_labels[col], fontsize=13)
    ax.set_ylabel('Density')
    ax.set_title(f'Posterior: {param_labels[col]}')
    if col == 2:
        ax.legend(fontsize=7)

plt.suptitle('Comparison of All Methods: Posterior Marginals', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/all_methods_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Quantitative Comparison Table

In [ ]:
print(f"{'Method':>25s}  {'beta_med':>9s} {'gamma_med':>10s} {'rho_med':>8s}  "
      f"{'beta_95w':>9s} {'gamma_95w':>10s} {'rho_95w':>8s}")
print('-' * 90)

for name, samples in methods.items():
    meds = [np.median(samples[:, k]) for k in range(3)]
    widths = []
    for k in range(3):
        lo, hi = np.percentile(samples[:, k], [2.5, 97.5])
        widths.append(hi - lo)
    print(f'{name:>25s}  {meds[0]:>9.4f} {meds[1]:>10.4f} {meds[2]:>8.4f}  '
          f'{widths[0]:>9.4f} {widths[1]:>10.4f} {widths[2]:>8.4f}')

## 4.6 Corner Plots for Each Method

In [ ]:
# Corner plots for each method
for name, samples in methods.items():
    fig_c = corner.corner(
        samples,
        labels=[r'$\beta$', r'$\gamma$', r'$\rho$'],
        quantiles=[0.16, 0.5, 0.84],
        show_titles=True,
        title_kwargs={'fontsize': 10},
        color=method_colors[list(methods.keys()).index(name)],
    )
    fig_c.suptitle(f'Posterior: {name}', fontsize=13, y=1.02)
    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '').replace('%', 'pct').replace('.', '')
    plt.savefig(f'../figures/corner_{safe_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4.7 Posterior Predictive Check (Best Method)

Simulate from the SMC-ABC posterior and overlay on observed data.

In [ ]:
# PPC from SMC-ABC particles (weighted resampling)
rng_ppc = np.random.default_rng(77)
n_ppc = 20
idx_ppc = rng_ppc.choice(len(smc_particles), size=n_ppc, p=smc_weights, replace=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
t_arr = np.arange(inf_ts.shape[1])

# Observed
for r in range(inf_ts.shape[0]):
    axes[0].plot(t_arr, inf_ts[r], color='steelblue', alpha=0.15)
    axes[1].plot(t_arr, rew_ts[r], color='seagreen', alpha=0.15)
axes[2].bar(np.arange(31), np.mean(deg_hist, axis=0), alpha=0.3, color='steelblue', label='Observed')

# Posterior predictive
for idx in idx_ppc:
    b, g, r = smc_particles[idx]
    inf_sim, rew_sim, deg_sim = simulate_fast(b, g, r, seed=int(rng_ppc.integers(0, 2**31)))
    axes[0].plot(t_arr, inf_sim, color='red', alpha=0.3, linewidth=0.8)
    axes[1].plot(t_arr, rew_sim, color='red', alpha=0.3, linewidth=0.8)

b, g, r = smc_particles[idx_ppc[0]]
_, _, deg_ppc = simulate_fast(b, g, r, seed=54321)
axes[2].bar(np.arange(31), deg_ppc, alpha=0.5, color='red', label='SMC posterior pred.')

axes[0].set_title('Infected fraction')
axes[1].set_title('Rewiring counts')
axes[2].set_title('Final degree histogram')
axes[2].legend()

from matplotlib.lines import Line2D
legend_el = [Line2D([0], [0], color='steelblue', alpha=0.5, label='Observed'),
             Line2D([0], [0], color='red', alpha=0.5, label='SMC-ABC posterior pred.')]
axes[0].legend(handles=legend_el, fontsize=9)

plt.suptitle('Posterior Predictive Check (SMC-ABC)', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/smc_ppc.png', dpi=150, bbox_inches='tight')
plt.show()